# FLUX Weight Injection — Complete Native Components

**Purpose:** Inject trained weights from phase checkpoints into Flux-Apex-V1.flx

**Target Version:** v8.1-complete

**Reference:** `output/flux_native_results/README.md`

## What This Notebook Does

1. **Download** phase checkpoints from HuggingFace (UnseenGAP/FLUX)
2. **Load** Flux-Apex-V1.flx (v8.0-autonomous)
3. **Audit** current component status (identify empty/config-only)
4. **Inject** trained weights from phases 1.5, 3, 4, 5, 6
5. **Verify** injected components have proper weight counts
6. **Save** as v8.1-complete

## Components to Inject

| Phase | Component | Current Status | Source Checkpoint |
|-------|-----------|----------------|-------------------|
| 1.5 | Causal Wave Chaining | Not present | `phase1.5.phase.pt` |
| 3 | Gravitational Relevance | Config only | `phase3.phase.pt` |
| 4 | Thermodynamic Learning | Config only | `phase4.phase.pt` |
| 5 | CGN Causal Graph | **EMPTY (0 params!)** | `phase5.phase.pt` |
| 6 | Three-Tier Memory | Metadata only | `phase6.phase.pt` |

After injection, ALL native FLUX components will have trained weights.

In [ ]:
# Cell 1: Setup and Imports
import sys
import os
import gc
import torch
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional

# Detect environment
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
ON_COLAB = 'COLAB_GPU' in os.environ

if ON_KAGGLE:
    # Clone repo if on Kaggle
    !git clone https://github.com/Unseengap/FLUX.git /kaggle/working/flux 2>/dev/null || git -C /kaggle/working/flux pull
    os.chdir('/kaggle/working/flux')
    ROOT = Path('/kaggle/working/flux')
elif ON_COLAB:
    !git clone https://github.com/Unseengap/FLUX.git /content/flux 2>/dev/null || git -C /content/flux pull
    os.chdir('/content/flux')
    ROOT = Path('/content/flux')
else:
    # Local environment
    ROOT = Path('.').resolve()
    if not (ROOT / 'flux_model.py').exists():
        ROOT = ROOT.parent
    os.chdir(ROOT)

sys.path.insert(0, str(ROOT))

print(f"✓ Working directory: {ROOT}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Download Checkpoints from HuggingFace
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO = "UnseenGAP/FLUX"
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

# Checkpoints we need to download
REQUIRED_CHECKPOINTS = [
    'checkpoints/Flux-Apex-V1.flx',  # Main model
    'checkpoints/phase1.phase.pt',    # CSE (already in .flx but good to have)
    'checkpoints/phase1.5.phase.pt',  # Causal Wave Chaining (note: dot not underscore)
    'checkpoints/phase2.phase.pt',    # Field (already in .flx)
    'checkpoints/phase3.phase.pt',    # Gravitational Relevance
    'checkpoints/phase4.phase.pt',    # Thermodynamic Learning
    'checkpoints/phase5.phase.pt',    # CGN Causal Graph
    'checkpoints/phase6.phase.pt',    # Three-Tier Memory
]

# Get HF token
HF_TOKEN = None
try:
    if ON_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    else:
        HF_TOKEN = os.environ.get('HF_TOKEN')
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded")
except:
    print("⚠ No HF_TOKEN — downloads may be rate-limited")

# List available files in repo
print(f"\n📂 Listing files in {HF_REPO}...")
try:
    files = list_repo_files(HF_REPO, token=HF_TOKEN)
    checkpoint_files = [f for f in files if f.startswith('checkpoints/') and (f.endswith('.pt') or f.endswith('.flx'))]
    print(f"Available checkpoints: {len(checkpoint_files)}")
    for f in checkpoint_files:
        print(f"  {f}")
except Exception as e:
    print(f"⚠ Could not list files: {e}")

# Download required checkpoints
print(f"\n📥 Downloading checkpoints...")
downloaded = {}
for ckpt_path in REQUIRED_CHECKPOINTS:
    local_path = ROOT / ckpt_path
    if local_path.exists():
        size_mb = local_path.stat().st_size / 1e6
        print(f"  ✓ {ckpt_path} (cached, {size_mb:.1f} MB)")
        downloaded[ckpt_path] = local_path
    else:
        try:
            path = hf_hub_download(
                repo_id=HF_REPO,
                filename=ckpt_path,
                local_dir=ROOT,
                token=HF_TOKEN,
            )
            size_mb = Path(path).stat().st_size / 1e6
            print(f"  ✓ {ckpt_path} ({size_mb:.1f} MB)")
            downloaded[ckpt_path] = Path(path)
        except Exception as e:
            print(f"  ✗ {ckpt_path}: {e}")

print(f"\n✓ Downloaded {len(downloaded)}/{len(REQUIRED_CHECKPOINTS)} checkpoints")

In [ ]:
# Cell 3: Load Flux-Apex Model and Audit Current State
from flux_model import FLUXModel

APEX_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
print(f"📂 Loading {APEX_PATH}...")

model = FLUXModel.load(str(APEX_PATH))

print(f"\n{'='*60}")
print(f"FLUX-APEX-V1 CURRENT STATE")
print(f"{'='*60}")
print(f"Version: {model.version}")
print(f"Phase: {model.phase}")
print(f"Format: {model.state.get('format', 'FLUX')}")

# Count parameters per component
def count_params(obj, depth=0):
    """Recursively count parameters in a nested dict/tensor structure."""
    if isinstance(obj, torch.Tensor):
        return obj.numel()
    if isinstance(obj, dict):
        total = 0
        for k, v in obj.items():
            total += count_params(v, depth+1)
        return total
    if isinstance(obj, (list, tuple)):
        return sum(count_params(x, depth+1) for x in obj)
    return 0

print(f"\n📊 Component Parameter Counts:")
print(f"{'Component':<25} {'Params':>15} {'Status':<20}")
print("-" * 60)

components_status = {}
for key in sorted(model.state.keys()):
    if key in ['format', 'version', 'phase', 'timestamp', 'runtime_config', 
               'components', 'metadata', 'can_continue_learning', 'modified',
               'modified_components', 'removed_components', 'state']:
        continue
    
    params = count_params(model.state.get(key, {}))
    
    # Determine status
    if params == 0:
        status = "❌ EMPTY"
    elif params < 1000:
        status = "⚠️ Config only"
    else:
        status = "✅ Has weights"
    
    components_status[key] = {'params': params, 'status': status}
    print(f"{key:<25} {params:>15,} {status:<20}")

# Identify what needs injection
print(f"\n{'='*60}")
print(f"COMPONENTS NEEDING INJECTION")
print(f"{'='*60}")

INJECTION_TARGETS = {
    'causal_wave_chaining': 'phase1.5.phase.pt',
    'gravity': 'phase3.phase.pt',
    'thermodynamic': 'phase4.phase.pt',
    'causal': 'phase5.phase.pt',
    'memory': 'phase6.phase.pt',
}

for comp, ckpt in INJECTION_TARGETS.items():
    status = components_status.get(comp, {}).get('status', '❓ Missing')
    params = components_status.get(comp, {}).get('params', 0)
    print(f"  {comp}: {params:,} params → inject from {ckpt}")

In [ ]:
# Cell 4: Inspect Phase Checkpoints
print(f"{'='*60}")
print(f"PHASE CHECKPOINT CONTENTS")
print(f"{'='*60}")

def inspect_checkpoint(path: Path) -> Dict[str, Any]:
    """Inspect a checkpoint and return key info."""
    if not path.exists():
        return {'error': 'File not found'}
    
    ckpt = torch.load(str(path), map_location='cpu', weights_only=False)
    
    info = {
        'keys': list(ckpt.keys()),
        'params_by_key': {},
        'total_params': 0,
    }
    
    for key in ckpt.keys():
        params = count_params(ckpt[key])
        info['params_by_key'][key] = params
        info['total_params'] += params
    
    return info

for ckpt_name in ['phase1.5.phase.pt', 'phase3.phase.pt', 'phase4.phase.pt', 
                   'phase5.phase.pt', 'phase6.phase.pt']:
    path = CHECKPOINTS_DIR / ckpt_name
    print(f"\n📦 {ckpt_name}:")
    
    info = inspect_checkpoint(path)
    if 'error' in info:
        print(f"  ✗ {info['error']}")
        continue
    
    print(f"  Total params: {info['total_params']:,}")
    print(f"  Keys: {info['keys']}")
    for key, params in info['params_by_key'].items():
        if params > 0:
            print(f"    {key}: {params:,} params")

In [ ]:
# Cell 5: Inject Phase 1.5 — Causal Wave Chaining
print(f"\n{'='*60}")
print(f"INJECTING PHASE 1.5: Causal Wave Chaining")
print(f"{'='*60}")

ckpt_path = CHECKPOINTS_DIR / 'phase1.5.phase.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    
    # Extract CWC state
    cwc_state = None
    cwc_config = None
    
    # Try different possible key structures
    for key in ['cwc_state_dict', 'chainer_state_dict', 'state_dict', 'cwc']:
        if key in ckpt:
            cwc_state = ckpt[key]
            print(f"  Found weights in key: {key}")
            break
    
    if cwc_state is None and 'config' in ckpt:
        # Might be nested differently
        for key in ckpt.keys():
            if isinstance(ckpt[key], dict) and 'state_dict' in ckpt[key]:
                cwc_state = ckpt[key]['state_dict']
                cwc_config = ckpt[key].get('config', {})
                print(f"  Found nested weights in key: {key}")
                break
    
    if cwc_state:
        cwc_params = count_params(cwc_state)
        print(f"  CWC parameters: {cwc_params:,}")
        
        # Add to model
        model.add_component(
            name='causal_wave_chaining',
            state_dict=cwc_state,
            config=cwc_config or ckpt.get('config', {}),
        )
        print(f"  ✓ Injected causal_wave_chaining")
    else:
        print(f"  ⚠ Could not find CWC weights, storing full checkpoint")
        model.state['causal_wave_chaining'] = {
            'state_dict': ckpt,
            'config': ckpt.get('config', {}),
            'source': 'phase1_5.phase.pt',
        }
else:
    print(f"  ✗ Checkpoint not found: {ckpt_path}")

In [ ]:
# Cell 6: Inject Phase 3 — Gravitational Relevance
print(f"\n{'='*60}")
print(f"INJECTING PHASE 3: Gravitational Relevance")
print(f"{'='*60}")

ckpt_path = CHECKPOINTS_DIR / 'phase3.phase.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    
    # Extract GR state (look for phase3_gr_state, phase3_decoder_state, etc.)
    gr_state = {}
    
    # FIXED: Use actual key names from phase3 checkpoint
    for key in ['phase3_gr_state', 'phase3_decoder_state', 'gravity_state_dict', 
                'gr_state_dict', 'gravity', 'gr', 'mass_tracker', 'spatial_index', 'state_dict']:
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            print(f"  Found {key}: {params:,} params")
            gr_state[key] = content
    
    if gr_state:
        total_params = count_params(gr_state)
        print(f"  Total GR parameters: {total_params:,}")
        
        # Update or create gravity component
        if 'gravity' not in model.state:
            model.state['gravity'] = {}
        
        model.state['gravity'].update({
            'state_dict': gr_state,
            'config': ckpt.get('config', {}),
            'source': 'phase3.phase.pt',
            'injected_at': datetime.now().isoformat(),
        })
        model.components['gravity'] = True
        print(f"  ✓ Injected gravity component")
    else:
        # Store full checkpoint
        print(f"  Storing full phase3 checkpoint")
        model.state['gravity'] = {
            'full_checkpoint': ckpt,
            'source': 'phase3.phase.pt',
        }
else:
    print(f"  ✗ Checkpoint not found: {ckpt_path}")

In [ ]:
# Cell 7: Inject Phase 4 — Thermodynamic Learning
print(f"\n{'='*60}")
print(f"INJECTING PHASE 4: Thermodynamic Learning")
print(f"{'='*60}")

ckpt_path = CHECKPOINTS_DIR / 'phase4.phase.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    
    # Extract TL state
    tl_state = {}
    
    # FIXED: Use actual key name from phase4 checkpoint (tl_state)
    for key in ['tl_state', 'thermodynamic_state_dict', 'tl_state_dict', 'thermodynamic', 
                'online_learner', 'temperature_state', 'energy_state', 'state_dict']:
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            print(f"  Found {key}: {params:,} params")
            tl_state[key] = content
    
    if tl_state:
        total_params = count_params(tl_state)
        print(f"  Total TL parameters: {total_params:,}")
        
        if 'thermodynamic' not in model.state:
            model.state['thermodynamic'] = {}
        
        model.state['thermodynamic'].update({
            'state_dict': tl_state,
            'config': ckpt.get('config', {}),
            'source': 'phase4.phase.pt',
            'injected_at': datetime.now().isoformat(),
        })
        model.components['thermodynamic'] = True
        print(f"  ✓ Injected thermodynamic component")
    else:
        model.state['thermodynamic'] = {
            'full_checkpoint': ckpt,
            'source': 'phase4.phase.pt',
        }
else:
    print(f"  ✗ Checkpoint not found: {ckpt_path}")

In [ ]:
# Cell 8: Inject Phase 5 — CGN Causal Graph (CRITICAL)
print(f"\n{'='*60}")
print(f"INJECTING PHASE 5: CGN Causal Graph (CRITICAL)")
print(f"{'='*60}")

ckpt_path = CHECKPOINTS_DIR / 'phase5.phase.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    
    # Extract CGN state — this is the CRITICAL missing component
    cgn_state = {}
    
    # FIXED: cgn_state has 14.7M params, causal_graph_state has 0 — check cgn_state FIRST!
    for key in ['cgn_state', 'tl_state', 'cgn_state_dict', 'coordinator_state_dict',
                'cgn', 'causal_graph', 'causal', 'manifold_state', 'state_dict']:
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            if params > 0:  # Only include keys with actual weights
                print(f"  Found {key}: {params:,} params")
                cgn_state[key] = content
            else:
                print(f"  Skipping {key}: 0 params")
    
    if cgn_state:
        total_params = count_params(cgn_state)
        print(f"  Total CGN parameters: {total_params:,}")
        
        # Update the causal component (currently empty!)
        model.state['causal'] = {
            'state_dict': cgn_state,
            'config': ckpt.get('config', {}),
            'cgn_nodes': ckpt.get('cgn_nodes', 56),
            'causal_links': ckpt.get('causal_links', 463),
            'source': 'phase5.phase.pt',
            'injected_at': datetime.now().isoformat(),
        }
        model.components['causal'] = True
        model.components['causal_tracker'] = True
        print(f"  ✓ Injected causal (CGN) component — was EMPTY, now has {total_params:,} params!")
    else:
        model.state['causal'] = {
            'full_checkpoint': ckpt,
            'source': 'phase5.phase.pt',
        }
        print(f"  ⚠ Stored full checkpoint (weights not in expected structure)")
else:
    print(f"  ✗ Checkpoint not found: {ckpt_path}")

In [ ]:
# Cell 9: Inject Phase 6 — Three-Tier Memory (CRITICAL)
print(f"\n{'='*60}")
print(f"INJECTING PHASE 6: Three-Tier Memory (CRITICAL)")
print(f"{'='*60}")

ckpt_path = CHECKPOINTS_DIR / 'phase6.phase.pt'
if ckpt_path.exists():
    ckpt = torch.load(str(ckpt_path), map_location='cpu', weights_only=False)
    
    print(f"Checkpoint keys: {list(ckpt.keys())}")
    
    # Extract Memory state — working, episodic, semantic
    memory_state = {}
    
    # FIXED: Use actual key names from phase6 checkpoint (*_state suffix)
    for key in ['working_memory_state', 'episodic_memory_state', 'semantic_memory_state', 
                'router_state', 'working_memory', 'episodic_memory', 'semantic_memory', 
                'memory_router', 'consolidation', 'memory', 'state_dict']:
        if key in ckpt:
            content = ckpt[key]
            params = count_params(content)
            print(f"  Found {key}: {params:,} params")
            memory_state[key] = content
    
    if memory_state:
        total_params = count_params(memory_state)
        print(f"  Total Memory parameters: {total_params:,}")
        
        # Update memory component
        if 'memory' not in model.state:
            model.state['memory'] = {}
        
        model.state['memory'].update({
            'state_dict': memory_state,
            'config': ckpt.get('config', {}),
            # FIXED: Use actual key names with _state suffix
            'working': memory_state.get('working_memory_state', memory_state.get('working_memory', {})),
            'episodic': memory_state.get('episodic_memory_state', memory_state.get('episodic_memory', {})),
            'semantic': memory_state.get('semantic_memory_state', memory_state.get('semantic_memory', {})),
            'router': memory_state.get('router_state', {}),
            'source': 'phase6.phase.pt',
            'injected_at': datetime.now().isoformat(),
        })
        model.components['memory'] = True
        print(f"  ✓ Injected memory component")
    else:
        model.state['memory'] = {
            'full_checkpoint': ckpt,
            'source': 'phase6.phase.pt',
        }
        print(f"  ⚠ Stored full checkpoint (no recognized keys)")
else:
    print(f"  ✗ Checkpoint not found: {ckpt_path}")

In [ ]:
# Cell 10: Update Version and Metadata
print(f"\n{'='*60}")
print(f"UPDATING VERSION AND METADATA")
print(f"{'='*60}")

# Update version
old_version = model.version
model.version = '8.1-complete'
model.phase = 'complete'

# Update metadata
model.metadata['last_modified'] = datetime.now().isoformat()
model.metadata['modified_components'] = [
    'causal_wave_chaining',
    'gravity',
    'thermodynamic',
    'causal',
    'memory',
]
model.metadata['injection_date'] = datetime.now().isoformat()
model.metadata['injection_source'] = 'flux_weight_injection.ipynb'
model.metadata['previous_version'] = old_version

# Track all injections
model.metadata['injections'] = {
    'causal_wave_chaining': 'phase1.5.phase.pt',
    'gravity': 'phase3.phase.pt',
    'thermodynamic': 'phase4.phase.pt',
    'causal': 'phase5.phase.pt',
    'memory': 'phase6.phase.pt',
}

print(f"Version: {old_version} → {model.version}")
print(f"Phase: {model.phase}")
print(f"Modified components: {model.metadata['modified_components']}")

In [ ]:
# Cell 11: Verify Injection — Parameter Count Comparison
print(f"\n{'='*60}")
print(f"VERIFICATION: Parameter Count Comparison")
print(f"{'='*60}")

print(f"\n{'Component':<25} {'Before':>15} {'After':>15} {'Change':<15}")
print("-" * 70)

after_status = {}
for key in sorted(model.state.keys()):
    if key in ['format', 'version', 'phase', 'timestamp', 'runtime_config', 
               'components', 'metadata', 'can_continue_learning', 'modified',
               'modified_components', 'removed_components', 'state']:
        continue
    
    params = count_params(model.state.get(key, {}))
    after_status[key] = params
    
    before = components_status.get(key, {}).get('params', 0)
    
    if params > before:
        change = f"✅ +{params - before:,}"
    elif params == before:
        change = "—"
    else:
        change = f"⚠️ {params - before:,}"
    
    print(f"{key:<25} {before:>15,} {params:>15,} {change:<15}")

# Summary
print(f"\n{'='*60}")
print(f"INJECTION SUMMARY")
print(f"{'='*60}")

total_before = sum(components_status.get(k, {}).get('params', 0) for k in after_status.keys())
total_after = sum(after_status.values())

print(f"Total parameters before: {total_before:,}")
print(f"Total parameters after:  {total_after:,}")
print(f"Parameters added:        {total_after - total_before:,}")

In [ ]:
# Cell 12: Save Injected Model

# CRITICAL: On disk-constrained environments, delete original first
SAVE_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
BACKUP_PATH = CHECKPOINTS_DIR / 'Flux-Apex-V1-backup.flx'

print(f"\n{'='*60}")
print(f"SAVING INJECTED MODEL")
print(f"{'='*60}")

# Check disk space
import shutil
total, used, free = shutil.disk_usage(CHECKPOINTS_DIR)
free_gb = free / 1e9
original_size_gb = SAVE_PATH.stat().st_size / 1e9 if SAVE_PATH.exists() else 0
print(f"Free disk space: {free_gb:.1f} GB")
print(f"Original model size: {original_size_gb:.1f} GB")

# Estimate actual model size (base 7.4B params + injected ~1B params = ~17GB)
ESTIMATED_SAVE_SIZE_GB = 17.0
print(f"Estimated save size: ~{ESTIMATED_SAVE_SIZE_GB:.1f} GB")

# Need estimated size + 1GB overhead to safely save
MIN_REQUIRED_GB = ESTIMATED_SAVE_SIZE_GB + 1.0

if free_gb < MIN_REQUIRED_GB:
    print(f"⚠ Low disk space ({free_gb:.1f} GB < {MIN_REQUIRED_GB:.1f} GB required)")
    
    # Step 1: Delete original .flx if it exists
    if SAVE_PATH.exists():
        print(f"  Deleting original .flx to make room...")
        os.remove(SAVE_PATH)
        gc.collect()
    
    # Step 2: Delete phase checkpoints (already loaded into memory)
    phase_checkpoints = [
        'phase1.phase.pt', 'phase1.5.phase.pt', 'phase2.phase.pt',
        'phase3.phase.pt', 'phase4.phase.pt', 'phase5.phase.pt', 'phase6.phase.pt'
    ]
    for ckpt_name in phase_checkpoints:
        ckpt_file = CHECKPOINTS_DIR / ckpt_name
        if ckpt_file.exists():
            size_mb = ckpt_file.stat().st_size / 1e6
            os.remove(ckpt_file)
            print(f"  Deleted {ckpt_name} ({size_mb:.1f} MB)")
    
    gc.collect()
    
    # Re-check disk space after cleanup
    total, used, free = shutil.disk_usage(CHECKPOINTS_DIR)
    free_gb = free / 1e9
    print(f"  After cleanup: {free_gb:.1f} GB free")
    
    if free_gb < MIN_REQUIRED_GB:
        raise RuntimeError(
            f"ABORT: Still only {free_gb:.1f} GB free after cleanup. "
            f"Need {MIN_REQUIRED_GB:.1f} GB to save safely. "
            f"Use Colab (40GB free) or clear more space."
        )

# Save
print(f"Saving to: {SAVE_PATH}")
model.save(str(SAVE_PATH), overwrite=True)

# Verify
if SAVE_PATH.exists():
    size_gb = SAVE_PATH.stat().st_size / 1e9
    print(f"✓ Saved: {SAVE_PATH.name} ({size_gb:.2f} GB)")
else:
    print(f"✗ Save failed!")

In [ ]:
# Cell 13: Final Verification — Reload and Check
print(f"\n{'='*60}")
print(f"FINAL VERIFICATION")
print(f"{'='*60}")

# Reload to verify
del model
gc.collect()

verify_model = FLUXModel.load(str(SAVE_PATH))

print(f"Version: {verify_model.version}")
print(f"Phase: {verify_model.phase}")

print(f"\n📊 Verified Component Parameters:")
print(f"{'Component':<25} {'Params':>15} {'Status':<20}")
print("-" * 60)

critical_ok = True
for key in ['cse', 'field', 'causal', 'memory', 'gravity', 'thermodynamic']:
    if key in verify_model.state:
        params = count_params(verify_model.state[key])
        status = "✅ Has weights" if params > 0 else "❌ EMPTY"
        if key in ['causal', 'memory'] and params == 0:
            critical_ok = False
        print(f"{key:<25} {params:>15,} {status:<20}")

print(f"\n{'='*60}")
if critical_ok:
    print("✅ ALL CRITICAL COMPONENTS HAVE WEIGHTS")
else:
    print("⚠️ SOME CRITICAL COMPONENTS STILL EMPTY")
print(f"{'='*60}")

In [ ]:
# Cell 14: Upload to HuggingFace (Optional)
UPLOAD_TO_HF = False  # Set to True to upload

if UPLOAD_TO_HF and HF_TOKEN:
    print(f"\n{'='*60}")
    print(f"UPLOADING TO HUGGINGFACE")
    print(f"{'='*60}")
    
    from huggingface_hub import HfApi
    
    api = HfApi(token=HF_TOKEN)
    
    # Use model.version (from Cell 10) if verify_model doesn't exist
    upload_version = verify_model.version if 'verify_model' in dir() else model.version
    
    api.upload_file(
        path_or_fileobj=str(SAVE_PATH),
        path_in_repo="checkpoints/Flux-Apex-V1.flx",
        repo_id=HF_REPO,
        commit_message=f"Inject trained weights - v{upload_version}",
    )
    
    print(f"✓ Uploaded to {HF_REPO}")
    print(f"  https://huggingface.co/{HF_REPO}")
else:
    print(f"\n⚠ Upload skipped (UPLOAD_TO_HF={UPLOAD_TO_HF})")
    print(f"  Set UPLOAD_TO_HF = True in Cell 14 to upload")

## Summary

After running this notebook:

1. **Phase 1.5 (CWC)** — Causal Wave Chaining weights injected
2. **Phase 3 (GR)** — Gravitational Relevance weights injected  
3. **Phase 4 (TL)** — Thermodynamic Learning weights injected
4. **Phase 5 (CGN)** — Causal Geometry Nodes weights injected (was EMPTY!)
5. **Phase 6 (Memory)** — Three-Tier Memory weights injected

The model is now **v8.1-complete** with ALL native FLUX components having trained weights.

### Next Steps

1. Run on Kaggle to perform the injection with GPU
2. Verify all tests pass with injected components
3. Upload to HuggingFace when ready